# Chương 4.2 — Sensitivity Analysis của Cụm B: Composite Blend (v2)

5 trọng số composite: w_reputation, w_adoption, w_services, w_publisher, w_compliance — ràng buộc sum=1.

Output từ `sensitivity bench --cluster=B`. Đổi `SNAP_ID` ở cell dưới nếu chạy trên snapshot khác.

> **Lưu ý:** Tier1Total/Tier2Total (trọng số tier của compliance) KHÔNG được khảo sát ở đây — chúng là input của ComputeComplianceScore (quyết định mức đóng góp của từng trường card *khi tính* điểm), không phải hệ số nhân hậu kỳ. Snapshot chỉ lưu ComplianceScore đã tính nên không thể tái lập trung thực; cần snapshot lưu raw ComplianceInput mới khảo sát được tier.

> **Lưu ý phương pháp:** vì 5 trọng số chịu ràng buộc tổng = 1 (recompute chuẩn hoá lại sau mỗi lần đổi), chúng KHÔNG độc lập. Do đó chỉ số Sobol (`sobol.csv`) bị lệch — đôi khi cho ST < S1 vốn không thể xảy ra với input độc lập. Vì vậy cụm B dùng **Dirichlet simplex sampling** (lấy mẫu đều trên đơn hình trọng số) làm phương pháp chính, kèm OAT/Tornado cho từng tham số; Sobol chỉ để tham khảo.

In [ ]:
import sys
sys.path.insert(0, '..')
from lib import setup_thesis_style, save_figure, load_oat, load_tornado, ternary_scatter

import matplotlib.pyplot as plt
import pandas as pd

setup_thesis_style()

SNAP_ID = "snap_20260613_185128"  # snapshot ID from `sensitivity snapshot list`
OUTPUT_DIR = f"../output/{SNAP_ID}"
CLUSTER = "B"

In [ ]:
simplex = pd.read_csv(f"{OUTPUT_DIR}/cluster_B/simplex.csv")
# 4-weight simplex → project to 3 corners (rep, svc, pub); w_compliance is the
# dropped 4th dim. mpltern renormalises each point, so the 3 shown coords are relative.
fig = ternary_scatter(
    simplex,
    t_col="w_reputation",
    l_col="w_services",
    r_col="w_publisher",
    color_col="spearman",
    title="Fig 4.2.1 — Spearman ρ trên đơn hình (w_rep, w_svc, w_pub)",
)
save_figure(fig, "4_2_1_ternary_spearman")
plt.show()

In [ ]:
tor = load_tornado(OUTPUT_DIR, CLUSTER)
tor["total"] = tor["delta_low"] + tor["delta_high"]
tor = tor.sort_values("total")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(tor["param"], -tor["delta_low"], color="#5b9bd5", label="Δ low (−)")
ax.barh(tor["param"], tor["delta_high"], color="#ed7d31", label="Δ high (+)")
ax.axvline(0, color="black", lw=0.5)
ax.set_title("Fig 4.2.2 — Tornado: |Δ mean composite| theo tham số")
ax.set_xlabel("Δ điểm composite trung bình")
ax.legend(loc="lower right")
fig.tight_layout()
save_figure(fig, "4_2_2_tornado")
plt.show()

In [ ]:
oat = load_oat(OUTPUT_DIR, CLUSTER)
fig, ax = plt.subplots(figsize=(7, 4.5))
for p in oat["param"].unique():
    sub = oat[oat["param"] == p]
    ax.plot(sub["value"], sub["spearman"], marker="o", label=p)
ax.axhline(0.95, ls="--", color="grey", lw=0.5)
ax.set_xlabel("Giá trị trọng số (trước chuẩn hoá)")
ax.set_ylabel("Spearman ρ (rank stability)")
ax.set_title("Fig 4.2.3 — OAT: độ ổn định thứ hạng theo từng trọng số composite")
ax.legend()
fig.tight_layout()
save_figure(fig, "4_2_3_weight_rank_stability")
plt.show()

In [ ]:
cols = ["w_reputation", "w_services", "w_publisher", "w_compliance", "spearman", "mean"]
print("Top-10 cấu hình trọng số theo độ ổn định thứ hạng (Spearman ρ):")
print(simplex.nlargest(10, "spearman")[cols].to_string(index=False))
print("\nTop-10 theo điểm composite trung bình:")
print(simplex.nlargest(10, "mean")[cols].to_string(index=False))